# Mix Ratios: Blend `think-midtrain` (Stage 5) with `think-dataset-clean-1930s`

Blends [`zachnorton03/think-midtrain/stage5/data`](https://huggingface.co/datasets/zachnorton03/think-midtrain/tree/main/stage5)
(midtrain) and [`jbduran/think-dataset-clean-1930s`](https://huggingface.co/datasets/jbduran/think-dataset-clean-1930s/tree/main)
(old corpus) at a chosen document-count ratio, writing **text-only** shards to
`mixed/ratio_<pct>/` in the midtrain repo. One ratio per run; each lives in its own folder.

Flow: index each source (`shard, row, char_len` per doc) → sample to the target ratio and shuffle
both sources together → plan output shards from `char_len` alone → pack by streaming one source
shard at a time → write manifest + provenance README.

`RANDOM_SEED` makes the whole selection reproducible; reruns skip output shards already uploaded.
The old corpus is only indexed until the pool reaches `OLD_DOCS_NEEDED * OLD_INDEX_OVERSAMPLE_FACTOR`
(seeded-random shard order), which avoids downloading all ~24 GB for a modest target at the cost
of not being a perfectly-uniform draw over all 328 shards.


## 0. Setup: dependencies + HF connection


In [ ]:
%pip -q install -U huggingface_hub pyarrow tqdm numpy

import os, json, re, time, random, tempfile, shutil
from collections import defaultdict

import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
import tqdm
from huggingface_hub import (
    HfApi, hf_hub_download, list_repo_files,
    CommitOperationAdd, CommitOperationDelete,
)
from huggingface_hub.utils import HfHubHTTPError
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN_Z')
api = HfApi(token=HF_TOKEN)

MIDTRAIN_REPO = 'zachnorton03/think-midtrain'   # destination repo AND midtrain source
OLD_REPO = 'jbduran/think-dataset-clean-1930s'   # old/original source (public, read-only)
MIDTRAIN_SRC_PREFIX = 'stage5/data'              # midtrain source shards

os.environ.setdefault('HF_HUB_DOWNLOAD_TIMEOUT', '30')
DL_MAX_RETRIES = 5
DL_ETAG_TIMEOUT = 30


def list_repo_paths(repo_id, token=None):
    return set(list_repo_files(repo_id=repo_id, repo_type='dataset', token=token))


def robust_hf_download(path_in_repo, repo_id, token=None):
    """hf_hub_download with a bounded per-attempt timeout + retry, so a hung CDN node raises
    instead of blocking forever."""
    last_exc = None
    for attempt in range(1, DL_MAX_RETRIES + 1):
        try:
            return hf_hub_download(
                repo_id=repo_id, repo_type='dataset', filename=path_in_repo,
                token=token, etag_timeout=DL_ETAG_TIMEOUT, force_download=(attempt > 1),
            )
        except Exception as exc:
            last_exc = exc
            print(f"  download retry {attempt}/{DL_MAX_RETRIES} for {repo_id}/{path_in_repo}: {exc}")
    raise last_exc


def robust_commit(operations, commit_message, max_retries=6):
    """api.create_commit with backoff that honors HF's 'Retry after N seconds' on 429s.

    One create_commit == one repo commit, no matter how many files it carries -- this is what
    keeps us under the 128-commits/hour limit. On a 429 we sleep for the server-advised delay
    (parsed from the error text) and retry, so a batch that trips the limit resumes itself
    instead of dying."""
    for attempt in range(1, max_retries + 1):
        try:
            return api.create_commit(
                repo_id=MIDTRAIN_REPO, repo_type='dataset',
                operations=operations, commit_message=commit_message,
            )
        except HfHubHTTPError as exc:
            status = getattr(getattr(exc, 'response', None), 'status_code', None)
            if status != 429 or attempt == max_retries:
                raise
            m = re.search(r'[Rr]etry (?:this action |after )?in ?(\d+)\s*(second|minute)', str(exc))
            wait = int(m.group(1)) * (60 if m and m.group(2) == 'minute' else 1) if m else 60 * attempt
            wait += 5  # small cushion past the window boundary
            tqdm.tqdm.write(f"  429 rate-limited; sleeping {fmt_secs(wait)} before retry "
                            f"{attempt + 1}/{max_retries}...")
            time.sleep(wait)
    raise RuntimeError("robust_commit exhausted retries")  # unreachable


def read_text_column(local_parquet):
    return pq.read_table(local_parquet, columns=['text']).column('text').to_pylist()


def fmt_secs(s):
    m, s = divmod(int(s), 60)
    h, m = divmod(m, 60)
    return f"{h}h{m:02d}m" if h else f"{m}m{s:02d}s"

## 1. Configuration -- edit these for each ratio, then run top to bottom


In [ ]:
RATIO_MIDTRAIN = 0.60          # fraction of the merged corpus from midtrain (0.0 - 1.0)
TARGET_TOTAL_DOCS = 50_000     # total document count of the merged output
RANDOM_SEED = 42               # fixes sampling + shuffle + shard-visit order (reproducible)
CHARS_PER_SHARD = 250_000_000  # same shard-size convention as Stages 2-5

# Old-corpus pool size = OLD_DOCS_NEEDED * this. Larger = more representative, more downloads.
OLD_INDEX_OVERSAMPLE_FACTOR = 1.5

# Output shards per HF commit. HF caps dataset commits at 128/hour; one create_commit uploads
# many files as a SINGLE commit, so batching keeps us well under the cap. ~25 means a full
# 109-shard run is ~5 commits instead of ~110.
SHARDS_PER_COMMIT = 25

# Peak packing RAM is bounded by how many OUTPUT shards are buffered at once (see section 7),
# NOT by CHARS_PER_SHARD. The packer works in passes holding <= this many chars of text; lower
# it if you still OOM, raise it (fewer passes = less re-parsing) if you have RAM to spare.
PACK_RAM_BUDGET_CHARS = 1_000_000_000   # ~1 GB of text buffered per pass

# --- Derived (no need to edit) ---
assert 0.0 <= RATIO_MIDTRAIN <= 1.0, "RATIO_MIDTRAIN must be in [0, 1]."
RATIO_PCT = round(RATIO_MIDTRAIN * 100)
DST_PREFIX = f'mixed/ratio_{RATIO_PCT}'
DATA_PREFIX = f'{DST_PREFIX}/data'
MIDTRAIN_DOCS_NEEDED = round(TARGET_TOTAL_DOCS * RATIO_MIDTRAIN)
OLD_DOCS_NEEDED = TARGET_TOTAL_DOCS - MIDTRAIN_DOCS_NEEDED
OUT_SHARDS_PER_PASS = max(1, PACK_RAM_BUDGET_CHARS // CHARS_PER_SHARD)  # output shards buffered per pass

print(f"Ratio: {RATIO_PCT}% midtrain / {100 - RATIO_PCT}% old-corpus")
print(f"Target total docs: {TARGET_TOTAL_DOCS:,}  ->  midtrain {MIDTRAIN_DOCS_NEEDED:,} / old {OLD_DOCS_NEEDED:,}")
print(f"Output: {MIDTRAIN_REPO}/{DST_PREFIX}/  |  seed: {RANDOM_SEED}  |  {SHARDS_PER_COMMIT} shard(s)/commit")
print(f"Packing: <= {OUT_SHARDS_PER_PASS} output shard(s)/pass (~{OUT_SHARDS_PER_PASS * CHARS_PER_SHARD / 1e9:.2f}B-char RAM ceiling)")

## 1b. (Optional) Wipe everything under this ratio's folder

Deletes **all** files under `mixed/ratio_<pct>/` (shards, manifest, README) in a single commit, so
you can rebuild the ratio from scratch. Uses the `RATIO_PCT` set in the config cell above -- run
that first. Guarded by `CONFIRM_WIPE`: set it to the exact ratio folder name to actually delete;
leave it `None` to just preview what would go.

In [ ]:
# Set CONFIRM_WIPE to the exact folder name (e.g. f'ratio_{RATIO_PCT}') to delete; None = preview only.
CONFIRM_WIPE = None

wipe_prefix = f'{DST_PREFIX}/'   # everything under mixed/ratio_<pct>/
to_delete = sorted(f for f in list_repo_paths(MIDTRAIN_REPO, token=HF_TOKEN) if f.startswith(wipe_prefix))

print(f"{len(to_delete)} file(s) under {MIDTRAIN_REPO}/{wipe_prefix}")
for f in to_delete[:10]:
    print(f"  {f}")
if len(to_delete) > 10:
    print(f"  ... and {len(to_delete) - 10} more")

if not to_delete:
    print("Nothing to wipe.")
elif CONFIRM_WIPE != f'ratio_{RATIO_PCT}':
    print(f"\nPREVIEW ONLY -- set CONFIRM_WIPE = 'ratio_{RATIO_PCT}' and rerun to delete these.")
else:
    # One create_commit deletes all of them together = a single commit against the hourly cap.
    robust_commit(
        [CommitOperationDelete(path_in_repo=f) for f in to_delete],
        commit_message=f'Wipe ratio_{RATIO_PCT} ({len(to_delete)} files)',
    )
    print(f"\nDeleted {len(to_delete)} file(s) from {MIDTRAIN_REPO}/{wipe_prefix}")

## 2. Index midtrain candidates (full -- small source)


In [ ]:
def index_source(shards, repo_id, token, source_tag, stop_at=None, shard_order=None, desc="Indexing"):
    """Index [{source, shard_path, row_idx, char_len}, ...], stopping at `stop_at` candidates
    (None = index all). Returns (index, shards_touched)."""
    order = shard_order if shard_order is not None else shards
    index, touched = [], []
    for shard_path in tqdm.tqdm(order, desc=desc):
        if stop_at is not None and len(index) >= stop_at:
            break
        local = robust_hf_download(shard_path, repo_id=repo_id, token=token)
        for row_idx, text in enumerate(read_text_column(local)):
            index.append({
                'source': source_tag, 'shard_path': shard_path, 'row_idx': row_idx,
                'char_len': len(text) if isinstance(text, str) else 0,
            })
        touched.append(shard_path)
    return index, touched


midtrain_shards = sorted(
    f for f in list_repo_paths(MIDTRAIN_REPO, token=HF_TOKEN)
    if f.startswith(f'{MIDTRAIN_SRC_PREFIX}/') and f.endswith('.parquet')
)
assert midtrain_shards, f"No shards under {MIDTRAIN_REPO}/{MIDTRAIN_SRC_PREFIX}/"

midtrain_index, _ = index_source(midtrain_shards, MIDTRAIN_REPO, HF_TOKEN, 'midtrain',
                                 desc="Indexing midtrain")
print(f"Midtrain pool: {len(midtrain_index):,} docs across {len(midtrain_shards)} shard(s).")
if len(midtrain_index) < MIDTRAIN_DOCS_NEEDED:
    print(f"WARNING: midtrain pool ({len(midtrain_index):,}) < needed ({MIDTRAIN_DOCS_NEEDED:,}); "
          f"all midtrain docs will be used and the output will fall short of TARGET_TOTAL_DOCS.")


## 3. Index old-corpus candidates (partial -- stop once oversampled)


In [ ]:
old_shards_all = sorted(f for f in list_repo_paths(OLD_REPO) if re.match(r'^shard_\d+\.parquet$', f))
assert old_shards_all, f"No shards at the root of {OLD_REPO}"
print(f"Old corpus: {len(old_shards_all)} shard(s) total.")

# Shard-visit order gets its own RNG, independent of sampling/shuffle.
old_shard_order = old_shards_all[:]
random.Random(RANDOM_SEED).shuffle(old_shard_order)

old_index_target = int(OLD_DOCS_NEEDED * OLD_INDEX_OVERSAMPLE_FACTOR)
old_index, old_shards_indexed = index_source(
    old_shards_all, OLD_REPO, None, 'old_corpus',
    stop_at=old_index_target, shard_order=old_shard_order, desc="Indexing old corpus",
)
print(f"Old-corpus pool: {len(old_index):,} docs across {len(old_shards_indexed)}/{len(old_shards_all)} "
      f"shard(s) (stopped once >= {old_index_target:,} candidates).")
if len(old_index) < OLD_DOCS_NEEDED:
    print(f"WARNING: old-corpus pool ({len(old_index):,}) < needed ({OLD_DOCS_NEEDED:,}) even after "
          f"indexing all {len(old_shards_all)} shards; the output will fall short of TARGET_TOTAL_DOCS.")


## 4. Sample each pool, then shuffle the two together

Each pool is sampled with its own RNG (so one source doesn't perturb the other's draw), then the
combined list is shuffled with a third -- all derived from `RANDOM_SEED`.


In [ ]:
def sample_pool(pool, n_needed, seed, label):
    if len(pool) <= n_needed:
        print(f"{label}: using all {len(pool):,} docs (< {n_needed:,} needed).")
        return list(pool)
    return random.Random(seed).sample(pool, n_needed)

midtrain_sample = sample_pool(midtrain_index, MIDTRAIN_DOCS_NEEDED, RANDOM_SEED + 1, 'midtrain')
old_sample = sample_pool(old_index, OLD_DOCS_NEEDED, RANDOM_SEED + 2, 'old_corpus')

combined = midtrain_sample + old_sample
random.Random(RANDOM_SEED + 3).shuffle(combined)   # interleave the two sources

actual_total = len(combined)
n_mid_total = len(midtrain_sample)
actual_pct = 100.0 * n_mid_total / max(actual_total, 1)
print(f"Combined + shuffled: {actual_total:,} docs "
      f"({n_mid_total:,} midtrain [{actual_pct:.1f}%] / {len(old_sample):,} old [{100 - actual_pct:.1f}%])")


## 5. Plan output shards (deterministic, index-only)

Packs the shuffled docs into ~`CHARS_PER_SHARD`-char shards from `char_len` alone (no text read).
The plan depends only on the reproducible shuffle, so shard *N* always holds the same docs -- that
is what makes packing resumable.


In [ ]:
plan = []            # plan[k] = list of doc-index dicts for output shard k
cur, cur_chars = [], 0
for doc in combined:
    cur.append(doc)
    cur_chars += doc['char_len']
    if cur_chars >= CHARS_PER_SHARD:
        plan.append(cur)
        cur, cur_chars = [], 0
if cur:
    plan.append(cur)

# Tag each doc with its output-shard index for the streaming packer.
for k, docs in enumerate(plan):
    for d in docs:
        d['out_idx'] = k

print(f"Planned {len(plan)} output shard(s) from {actual_total:,} docs.")
for k, docs in enumerate(plan):
    n_mid = sum(1 for d in docs if d['source'] == 'midtrain')
    print(f"  shard_{k:05d}: {len(docs):,} docs ({n_mid:,} midtrain / {len(docs) - n_mid:,} old) | "
          f"{sum(d['char_len'] for d in docs):,} chars")


## 6. Build per-shard provenance from the plan

Built from the plan, not upload state, so the manifest/README describe the full set even on a
resume.


In [ ]:
def build_shard_records(plan):
    records = []
    for k, docs in enumerate(plan):
        counts = defaultdict(int)
        for d in docs:
            counts[(d['source'], d['shard_path'])] += 1
        n_mid = sum(1 for d in docs if d['source'] == 'midtrain')
        records.append({
            'shard': f'shard_{k:05d}.parquet',
            'n_docs': len(docs), 'n_midtrain': n_mid, 'n_old_corpus': len(docs) - n_mid,
            'chars': sum(d['char_len'] for d in docs),
            'source_shards': {f'{src}:{path}': cnt for (src, path), cnt in sorted(counts.items())},
        })
    return records

shard_records = build_shard_records(plan)
print(f"Provenance built for {len(shard_records)} planned shard(s).")

## 7. Pack + upload -- streaming, memory-bounded, resumable

Builds each output shard by reading its source shards and routing the selected docs into a
`position -> text` buffer, then writes + uploads it. Because the §4 shuffle scatters every output
shard's docs across ~all source shards, an output shard can't be finalized until its last straggler
arrives -- so buffering *all* output shards at once would hold nearly the **whole corpus** in RAM
(the cause of OOM crashes on large-doc runs; note it's independent of `CHARS_PER_SHARD`).

Instead we pack in **passes** of at most `OUT_SHARDS_PER_PASS` output shards, so peak RAM is
~`OUT_SHARDS_PER_PASS * CHARS_PER_SHARD` chars no matter how big the corpus is. Source shards are
re-read once per pass, but `hf_hub_download` caches them locally so that's a re-parse, not a
re-download. Finished shards upload in batches of `SHARDS_PER_COMMIT` (one HF commit each), and
already-uploaded shards are skipped so a rerun resumes where it left off.

Before trusting that skip, a **backwards-compat guard** reads back a sample of the shards already in
the folder and checks their doc count + per-row char lengths against the current plan. A shard the
old (pre-passes) code uploaded regenerates identically here, so it passes; but if the folder was
built with a different `RANDOM_SEED` / `TARGET_TOTAL_DOCS` / `RATIO_MIDTRAIN` / `CHARS_PER_SHARD`,
`shard_k` now means something else and the guard **aborts** rather than welding two mixes together.

In [ ]:
existing_out = {os.path.basename(f) for f in list_repo_paths(MIDTRAIN_REPO, token=HF_TOKEN)
               if f.startswith(f'{DATA_PREFIX}/') and f.endswith('.parquet')}
todo_out = [k for k in range(len(plan)) if f'shard_{k:05d}.parquet' not in existing_out]
print(f"Output shards: {len(plan)} planned | already uploaded: {len(plan) - len(todo_out)} | remaining: {len(todo_out)}")

# --- Backwards-compat guard: existing shards must match the CURRENT plan ---------------------
# Resume-skip trusts that an already-uploaded shard_k holds exactly plan[k]. That only holds if
# the folder was built with the same RANDOM_SEED / TARGET_TOTAL_DOCS / RATIO_MIDTRAIN /
# CHARS_PER_SHARD; otherwise the shuffle + packing differ and shard_k means something else.
# Verify a sample of existing shards against the plan BEFORE trusting the skip, so a changed
# param aborts loudly instead of silently welding two incompatible mixes into one folder.
existing_ks = sorted(int(m.group(1)) for f in existing_out
                     if (m := re.match(r'shard_(\d+)\.parquet$', f)))
orphans = [k for k in existing_ks if k >= len(plan)]
if orphans:
    raise RuntimeError(
        f"ABORT: {len(orphans)} existing shard(s) (e.g. shard_{orphans[0]:05d}.parquet) lie beyond the "
        f"current plan of {len(plan)} shard(s) -- this folder was built with different params. Wipe "
        f"mixed/ratio_{RATIO_PCT}/ (section 1b) and rebuild, or restore the params it was built with.")

def _plan_lens(k):   # per-row char lengths shard_k must have, in write (out_pos) order
    return [d['char_len'] for d in sorted(plan[k], key=lambda d: (d['source'], d['shard_path'], d['row_idx']))]

already_ks = [k for k in existing_ks if k < len(plan)]
if already_ks:
    check_ks = sorted({already_ks[0], already_ks[len(already_ks) // 2], already_ks[-1]})
    for k in check_ks:
        local = robust_hf_download(f'{DATA_PREFIX}/shard_{k:05d}.parquet', repo_id=MIDTRAIN_REPO, token=HF_TOKEN)
        got = [len(t) if isinstance(t, str) else 0 for t in read_text_column(local)]
        if got != _plan_lens(k):
            raise RuntimeError(
                f"ABORT: existing shard_{k:05d}.parquet ({len(got)} docs) does not match the current plan "
                f"({len(plan[k])} docs). This folder was built with different RANDOM_SEED / "
                f"TARGET_TOTAL_DOCS / RATIO_MIDTRAIN / CHARS_PER_SHARD. Restore those params to resume, "
                f"or wipe mixed/ratio_{RATIO_PCT}/ (section 1b) and rebuild.")
    print(f"Backwards-compat OK: existing shard(s) {check_ks} match the current plan; safe to resume.")

# Stable within-shard order (sort by source/shard/row) so streamed texts land deterministically.
# Index-only (no text) -> cheap even for millions of docs.
for k in todo_out:
    for pos, d in enumerate(sorted(plan[k], key=lambda d: (d['source'], d['shard_path'], d['row_idx']))):
        d['out_pos'] = pos

# Pack in passes of <= OUT_SHARDS_PER_PASS output shards so only that many are buffered in RAM at
# once (peak ~ OUT_SHARDS_PER_PASS * CHARS_PER_SHARD chars). Source shards are re-read once per
# pass, but hf_hub_download caches them locally -> a re-parse, not a re-download.
passes = [todo_out[i:i + OUT_SHARDS_PER_PASS] for i in range(0, len(todo_out), OUT_SHARDS_PER_PASS)]
print(f"Packing in {len(passes)} pass(es) of <= {OUT_SHARDS_PER_PASS} shard(s) "
      f"(~{OUT_SHARDS_PER_PASS * CHARS_PER_SHARD / 1e9:.2f}B-char RAM ceiling).")

pack_tmp = tempfile.mkdtemp()
pending_ops = []       # finished (path_in_repo, local_path) awaiting a batch commit
n_committed = 0

def commit_pending(force=False):
    """Commit buffered shard files as ONE HF commit once >= SHARDS_PER_COMMIT are ready
    (or force=True to flush the tail). Local temp files are removed after a successful commit."""
    global n_committed
    if not pending_ops or (len(pending_ops) < SHARDS_PER_COMMIT and not force):
        return
    ops = [CommitOperationAdd(path_in_repo=pir, path_or_fileobj=loc) for pir, loc in pending_ops]
    names = [os.path.basename(pir) for pir, _ in pending_ops]
    msg = (f'Mixed ratio_{RATIO_PCT}: {names[0]}..{names[-1]} ({len(ops)} shards)'
           if len(ops) > 1 else f'Mixed ratio_{RATIO_PCT} {names[0]}')
    robust_commit(ops, msg)
    for _, loc in pending_ops:
        os.remove(loc)
    n_committed += len(pending_ops)
    tqdm.tqdm.write(f"  committed {len(pending_ops)} shard(s) [{names[0]}..{names[-1]}] "
                    f"({n_committed}/{len(todo_out)} uploaded)")
    pending_ops.clear()

def stage_output_shard(k, buf):
    """Write output shard k from its {position: text} buffer to a temp parquet, queue for commit."""
    shard_name = f'shard_{k:05d}.parquet'
    texts = [buf[pos] for pos in range(len(plan[k]))]
    local_out = os.path.join(pack_tmp, shard_name)
    pq.write_table(pa.table({'text': texts}), local_out)
    pending_ops.append((f'{DATA_PREFIX}/{shard_name}', local_out))
    commit_pending()

t0 = time.time()
for pi, pass_ks in enumerate(passes, 1):
    # Source shards feeding this pass, and how many docs each output shard still needs.
    needed_by_source = defaultdict(list)   # (source, shard_path) -> [doc, ...]
    remaining_in_out = defaultdict(int)    # out_idx -> docs not yet collected
    for k in pass_ks:
        for d in plan[k]:
            needed_by_source[(d['source'], d['shard_path'])].append(d)
            remaining_in_out[k] += 1
    out_buffers = defaultdict(dict)        # out_idx -> {position: text}, this pass only
    for (source, shard_path) in tqdm.tqdm(sorted(needed_by_source),
                                          desc=f"Pass {pi}/{len(passes)} (stream by source shard)"):
        repo_id = MIDTRAIN_REPO if source == 'midtrain' else OLD_REPO
        token = HF_TOKEN if source == 'midtrain' else None
        text_col = read_text_column(robust_hf_download(shard_path, repo_id=repo_id, token=token))
        for d in needed_by_source[(source, shard_path)]:
            out_buffers[d['out_idx']][d['out_pos']] = text_col[d['row_idx']]
            remaining_in_out[d['out_idx']] -= 1
            if remaining_in_out[d['out_idx']] == 0:      # last doc arrived -> write + free it
                k = d['out_idx']
                stage_output_shard(k, out_buffers.pop(k))
                tqdm.tqdm.write(f"  wrote shard_{k:05d}.parquet")
        del text_col
    del out_buffers, needed_by_source, remaining_in_out   # release this pass's RAM before the next

commit_pending(force=True)   # flush the final partial batch
print(f"\nPacked + committed {n_committed} shard(s) in {fmt_secs(time.time() - t0)} to {MIDTRAIN_REPO}/{DATA_PREFIX}/")

## 8. Write the manifest + per-shard-provenance README


In [ ]:
total_docs = sum(r['n_docs'] for r in shard_records)
total_midtrain = sum(r['n_midtrain'] for r in shard_records)
total_old = sum(r['n_old_corpus'] for r in shard_records)
total_chars = sum(r['chars'] for r in shard_records)
actual_pct = 100.0 * total_midtrain / max(total_docs, 1)

manifest = {
    'ratio_pct_midtrain_target': RATIO_PCT,
    'ratio_pct_midtrain_actual': round(actual_pct, 2),
    'target_total_docs': TARGET_TOTAL_DOCS,
    'random_seed': RANDOM_SEED,
    'chars_per_shard': CHARS_PER_SHARD,
    'old_index_oversample_factor': OLD_INDEX_OVERSAMPLE_FACTOR,
    'midtrain_source': f'{MIDTRAIN_REPO}/{MIDTRAIN_SRC_PREFIX}',
    'old_corpus_source': OLD_REPO,
    'totals': {'n_shards': len(shard_records), 'n_docs': total_docs,
               'n_midtrain': total_midtrain, 'n_old_corpus': total_old, 'chars': total_chars},
    'shard_records': shard_records,
    'updated_at': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
}

meta_tmp = tempfile.mkdtemp()
manifest_local = os.path.join(meta_tmp, 'manifest.json')
with open(manifest_local, 'w') as f:
    json.dump(manifest, f, indent=2, sort_keys=True)


def source_shard_lines(source_shards):
    out = []
    for key, cnt in sorted(source_shards.items(), key=lambda kv: -kv[1]):
        src, path = key.split(':', 1)
        out.append(f"  - `{path}` ({src}): {cnt:,} doc(s)")
    return '\n'.join(out)

shard_sections = '\n'.join(
    f'''### `{r["shard"]}`

- docs: {r["n_docs"]:,} ({r["n_midtrain"]:,} midtrain / {r["n_old_corpus"]:,} old-corpus)
- chars: {r["chars"]:,}
- source shards:
{source_shard_lines(r["source_shards"])}
'''
    for r in shard_records
)

readme = f'''# Mixed corpus -- ratio_{RATIO_PCT} ({RATIO_PCT}% midtrain / {100 - RATIO_PCT}% old-corpus)

Blend of [`{MIDTRAIN_REPO}/{MIDTRAIN_SRC_PREFIX}`](https://huggingface.co/datasets/{MIDTRAIN_REPO}/tree/main/{MIDTRAIN_SRC_PREFIX})
and [`{OLD_REPO}`](https://huggingface.co/datasets/{OLD_REPO}/tree/main), produced by
`midtraining_mix_ratios.ipynb`. Output shards are **text-only** (`text` column), documents from
both sources shuffled together within each shard (not blocked old-then-new).

- Target ratio: **{RATIO_PCT}% midtrain / {100 - RATIO_PCT}% old-corpus**
- Target total docs: {TARGET_TOTAL_DOCS:,}
- Random seed: {RANDOM_SEED} (fixes sampling + shuffle + shard-visit order -- reproducible)
- Actual totals: {total_docs:,} docs ({total_midtrain:,} midtrain [{actual_pct:.1f}%] / {total_old:,} old-corpus [{100 - actual_pct:.1f}%]), {total_chars:,} chars, {len(shard_records)} shard(s)

## Per-shard provenance

Each shard lists which source shards (and how many docs from each) went into it.

{shard_sections}

Updated: {time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())}
'''

readme_local = os.path.join(meta_tmp, 'README.md')
with open(readme_local, 'w', encoding='utf-8') as f:
    f.write(readme)

# manifest.json + README.md go up together as a SINGLE commit (one more against the hourly cap).
robust_commit(
    [CommitOperationAdd(path_in_repo=f'{DST_PREFIX}/manifest.json', path_or_fileobj=manifest_local),
     CommitOperationAdd(path_in_repo=f'{DST_PREFIX}/README.md', path_or_fileobj=readme_local)],
    commit_message=f'Mixed ratio_{RATIO_PCT} manifest + README',
)

print(f"Uploaded manifest.json + README.md to {MIDTRAIN_REPO}/{DST_PREFIX}/")
print(f"Totals: {total_docs:,} docs ({actual_pct:.1f}% midtrain), {total_chars:,} chars, {len(shard_records)} shard(s).")

## 9. Read-back verification


In [ ]:
check = robust_hf_download(f'{DST_PREFIX}/manifest.json', repo_id=MIDTRAIN_REPO, token=HF_TOKEN)
with open(check) as f:
    m = json.load(f)
print(json.dumps({k: v for k, v in m.items() if k != 'shard_records'}, indent=2))

out_shards = sorted(f for f in list_repo_paths(MIDTRAIN_REPO, token=HF_TOKEN)
                    if f.startswith(f'{DATA_PREFIX}/') and f.endswith('.parquet'))
print(f"\nUploaded output shards: {len(out_shards)} / {len(plan)} planned.")
if out_shards:
    first = robust_hf_download(out_shards[0], repo_id=MIDTRAIN_REPO, token=HF_TOKEN)
    df = pq.read_table(first).to_pandas()
    print(f"First shard {os.path.basename(out_shards[0])}: {len(df):,} docs, columns={list(df.columns)}")
    lengths = df['text'].str.len()
    print(f"text length: min={lengths.min():,} median={int(lengths.median()):,} max={lengths.max():,}")
print(f"\nRatio {RATIO_PCT}% mix complete. Change RATIO_MIDTRAIN (and/or TARGET_TOTAL_DOCS) and rerun for another ratio.")
